In [1]:
from pathlib import Path

file_path = Path("/Users/omkarsatapaphy/python_works/Building_Language_models/toy_datasets/TinyStoriesV2-GPT4-train.txt")

with file_path.open("r", encoding="utf-8") as f:
    text = f.read()

print(f"Loaded {len(text):,} characters from {file_path.name}")
print(text[:1000])  # preview

Loaded 2,226,845,268 characters from TinyStoriesV2-GPT4-train.txt

Once upon a time there was a little boy named Ben. Ben loved to explore the world around him. He saw many amazing things, like beautiful vases that were on display in a store. One day, Ben was walking through the store when he came across a very special vase. When Ben saw it he was amazed!  
He said, “Wow, that is a really amazing vase! Can I buy it?” 
The shopkeeper smiled and said, “Of course you can. You can take it home and show all your friends how amazing it is!”
So Ben took the vase home and he was so proud of it! He called his friends over and showed them the amazing vase. All his friends thought the vase was beautiful and couldn't believe how lucky Ben was. 
And that's how Ben found an amazing vase in the store!
<|endoftext|>
Once upon a time, there was a reliable otter named Ollie. He lived in a river with his family. They all loved to play and swim together.
One day, Ollie's mom said, "Ollie, hurry and get so

In [2]:
import tiktoken
enc = tiktoken.get_encoding("gpt2") 

In [3]:
# ids = enc.encode(text, allowed_special={"<|endoftext|>"})
# print(f"Tokenized into {len(ids):,} tokens")


In [4]:
import os, numpy as np

SEP = "<|endoftext|>"
n_chunks = os.cpu_count() * 4          # a few big chunks per core, NOT millions

# split on the separator, then regroup docs into ~equal large chunks
docs = text.split(SEP)
target = len(text) // n_chunks
chunks, buf, size = [], [], 0
for d in docs:
    buf.append(d)
    size += len(d) + len(SEP)
    if size >= target:
        chunks.append(SEP.join(buf) + SEP)
        buf, size = [], 0
if buf:
    chunks.append(SEP.join(buf))

# encode the big chunks in parallel across every core
batches = enc.encode_batch(
    chunks,
    allowed_special={SEP},
    num_threads=os.cpu_count(),        # match cores; 100 just adds contention
)

# concatenate as numpy (fast, C-level) instead of a Python list comprehension
ids = np.concatenate([np.asarray(b, dtype=np.uint32) for b in batches])
print(f"Tokenized into {len(ids):,} tokens using {os.cpu_count()} cores")


Tokenized into 547,725,817 tokens using 14 cores


In [5]:
text_100M = text[:len(text) // 5]

def tokenize(text, n_chunks=None):
    docs = text.split(SEP)
    if n_chunks is None:
        n_chunks = os.cpu_count() * 4
    target = len(text) // n_chunks
    chunks, buf, size = [], [], 0
    for d in docs:
        buf.append(d)
        size += len(d) + len(SEP)
        if size >= target:
            chunks.append(SEP.join(buf) + SEP)
            buf, size = [], 0
    if buf:
        chunks.append(SEP.join(buf))
    batches = enc.encode_batch(
        chunks,
        allowed_special={SEP},
        num_threads=os.cpu_count(),
    )
    ids = np.concatenate([np.asarray(b, dtype=np.uint32) for b in batches])
    return ids

print(f"Tokenized into {len(tokenize(text_100M)):,} tokens using {os.cpu_count()} cores")


Tokenized into 109,556,301 tokens using 14 cores


In [6]:
out_path = Path("/Users/omkarsatapaphy/python_works/Building_Language_models/toy_datasets/tiny_stories_100M.txt")

with out_path.open("w", encoding="utf-8") as f:
    f.write(text_100M)

